# OrbitWatch: Clay features + heads
Runtime > L4 or A100 GPU. Run cells top to bottom; everything heavy is cached on Drive, so a disconnect only costs a re-run of the cheap cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/orbitwatch'
!mkdir -p {OUT}

In [ ]:
!rm -rf /content/Computer_vision
!git clone -q https://github.com/riya9900/Computer_vision.git
%cd /content/Computer_vision/training
!pip install -q -r requirements.txt
!pip install -q --no-deps git+https://github.com/Clay-foundation/model.git

## 1. Download Clay v1.5 once, keep encoder only (~1.2 GB) on Drive

In [ ]:
import os
ENC = f'{OUT}/clay_encoder_v1.5.pt'
if not os.path.exists(ENC):
    !python -m orbit_train.clay extract --out {ENC}

## 2. Cache frozen-Clay features (the only GPU-heavy step)

In [ ]:
!python -m orbit_train.cache_features oscd --encoder {ENC} --data /content/oscd --out {OUT}/features

In [ ]:
!python -m orbit_train.cache_features eurosat --encoder {ENC} --data /content/eurosat --out {OUT}/features

## 3. Train heads on cached features (minutes)

In [ ]:
for seed in (0, 1, 2):
    !python -m orbit_train.train_land_cover --cache {OUT}/features --out {OUT} --seed {seed}

In [ ]:
for head in ('conv', 'linear'):
    for seed in (0, 1, 2):
        !python -m orbit_train.train_change_clay --cache {OUT}/features --out {OUT} --head {head} --seed {seed}

## 4. Baseline and final tables

In [ ]:
!python -m orbit_train.baseline_spectral --root /content/oscd --out {OUT}

In [ ]:
!python -m orbit_train.results --dir {OUT}

In [ ]:
from IPython.display import Image
Image(f'{OUT}/eurosat_confusion.png')